# 05 — Feature Engineering & Thermodynamic Ratios

**Objective:** Engineer thermodynamic gas turbine features derived from the Brayton cycle:
- Pressure ratio: `(CDP * 1000) / AP`
- Expansion temperature drop: `TIT - TAT`
- Expansion temperature ratio: `TIT / TAT`
- Ambient air density proxy: `AP / (AT + 273.15)`
- Filter differential ratio: `(AFDP * 1000) / AP`
- Specific enthalpy proxy: `(TIT - TAT) * CDP`
- Track A specific: `specific_power` (`TEY / CDP`), `thermal_load_ratio` (`TEY / TIT`)
- Track B strict isolation: no TEY-derived features permitted.


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.features.feature_engineering import compute_domain_features, get_feature_names


## 1. Computing Thermodynamic Domain Features


In [2]:
master_df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")

df_feat_a = compute_domain_features(master_df, include_tey=True)
df_feat_b = compute_domain_features(master_df, include_tey=False)

features_a = get_feature_names("A", engineered=True)
features_b = get_feature_names("B", engineered=True)

print(f"Track A feature count (engineered): {len(features_a)}")
print(f"Track B feature count (engineered): {len(features_b)}")

manifest = {
    "track_a_features_raw": get_feature_names("A", engineered=False),
    "track_a_features_engineered": features_a,
    "track_b_features_raw": get_feature_names("B", engineered=False),
    "track_b_features_engineered": features_b,
    "total_observations": len(master_df)
}

out_manifest = ROOT / "data" / "processed" / "feature_manifest.json"
with open(out_manifest, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved feature manifest to {out_manifest.relative_to(ROOT)}")
print("Feature engineering pipeline COMPLETE.")


Track A feature count (engineered): 18
Track B feature count (engineered): 15
Saved feature manifest to data\processed\feature_manifest.json
Feature engineering pipeline COMPLETE.
